# Collecting circuit-quality measurements on Amazon Braket

**Professor’s runbook · Rigetti Cepheus-1-108Q and IonQ Forte-1.** Run this notebook from your working clone of the thesis repository. The default cells verify the environment, price the experiment and replay a previously recovered Rigetti result offline. Cloud and paid stages require explicit inputs.

Fez and Kingston supplied 2,409 training-collection jobs across 189 configurations, with 1,024 shots, optimization level 3 and seed 11. The analysis grouped configurations across both chips for repeated five-fold evaluation and held out later windows to study drift. A subsequent 42-configuration, two-repetition collection on each of Fez, Kingston and Marrakesh tested predictions frozen before execution. Marrakesh had no training rows. The AWS study keeps the same circuit factories, ideal distributions, success criteria and four targets (success, DSR, Hellinger fidelity and TVD). Every Braket task produces one observation.

[IBM model results](../report/ml_estimation/report.md) · [Prospective protocol](../docs/prospective_validation.md) · [Completed validation](../report/ml_estimation/prospective/validation-v1_summary.md)

## 1. Install from the repository and select the kernel

Clone the repository and check out the branch containing this notebook. From the clone’s top directory, on Linux, macOS or WSL, create a local environment with CPython 3.11.14. The campaign journal uses POSIX file locking. The repository directory and the run directory must be writable.

```sh
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -c docs/braket-requirements.txt -e '.[aws,torch]' ipykernel jupyterlab
python -m ipykernel install --user --name braket-thesis --display-name "Braket thesis"
python -m jupyterlab notebooks/braket_campaign.ipynb
```

Select **Braket thesis** as the kernel. The next cell compares the active kernel to the measured local experiment stack and stops on a mismatch. The printed revision identifies the code in this checkout. The operational cells call the same Python entry point as `python scripts/run.py braket`.

In [ ]:
import importlib.metadata as metadata
import os
import platform
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "docs/braket-requirements.txt").is_file())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
expected = {"Python": "3.11.14", **dict(line.split("==") for line in
            (ROOT / "docs/braket-requirements.txt").read_text().splitlines()
            if line and not line.startswith("#"))}
mismatches = []
for name, version in expected.items():
    try:
        actual = platform.python_version() if name == "Python" else metadata.version(name)
    except metadata.PackageNotFoundError:
        actual = "missing"
    print(f"{name:32s} expected {version:12s} installed {actual}")
    if actual != version:
        mismatches.append(name)
print("Kernel", sys.executable)
if (ROOT / ".git").exists():
    print("Repository revision", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
if mismatches:
    raise RuntimeError("Select the pinned kernel or reinstall with the constraints: " + ", ".join(mismatches))

## 2. Review the agreed QPU budget

One task is one circuit, submitted once. A repetition or a later window incurs another task fee. Use the calculator to see how task count and shot count affect the amount available for IonQ. AWS lists $0.30 per task and $0.000425 per shot for Cepheus, or $0.08 per shot for Forte. On-demand Forte has a 100-shot minimum. This design uses ordinary on-demand execution with error mitigation off. One mitigated Forte task requires at least 2,500 shots and costs $200.30. Verify prices on the day of execution. S3, tax and other account charges fall outside this QPU estimate.

The IBM `full` profile has 391 executions in one collection, and `validation-v1` adds 84. The six families in the static Braket runner account for 292 development and 76 validation executions. One pass over those families gives 146 development plus 38 validation slots, including known anchor circuits repeated in the later collection. Wide-profile figures are pricing scenarios and require conversion checks for every selected circuit.

[AWS prices](https://aws.amazon.com/braket/pricing/) · [Shot limits](https://docs.aws.amazon.com/braket/latest/developerguide/braket-quotas.html) · [Regions and ARNs](https://docs.aws.amazon.com/braket/latest/developerguide/braket-devices.html)

In [ ]:
from decimal import Decimal
import pandas as pd
from experiments import aws_campaign as aws, braket_study as study

def compare(tasks, shots, budget=study.CAP):
    cost = tasks * aws.task_price(aws.DEVICE_ARN, shots)
    balance = budget - cost
    return {"Rigetti tasks": tasks, "shots/task": shots, "Rigetti USD": float(cost),
            "balance USD": float(balance), "100-shot Forte tasks": max(0, int(balance // Decimal("8.30")))}

pd.DataFrame([compare(n, shots) for n, shots in
              [(391, 1024), (475, 1024), (475, 256), (184, 1024), (184, 512), (108, 1024)]])

**Interpreting the shot difference.** At an observed success rate near 50%, a 95% Wilson interval has a half-width of 9.6 percentage points for 100 shots, 4.3 for 512, and 3.1 for 1,024. Better fidelity can move the observed rate closer to one. The shot count still determines sampling precision, and sparse histograms also affect DSR, Hellinger fidelity and TVD. Report the interval per task. For a cross-device sensitivity analysis, subsample the saved Rigetti counts to 100 shots with a recorded seed and compare those metrics alongside the full-shot measurements.

In a retrospective shot-sensitivity check, we sampled without replacement 20 times per shot count from each of 1,838 archived 1,024-shot Fez and Kingston development histograms in these six families (NumPy seed 11). The mean absolute changes from the original histogram in success / TVD / Hellinger fidelity were 0.0091 / 0.0096 / 0.0091 at 512 shots and 0.0275 / 0.0294 / 0.0279 at 100. This measures sensitivity to fewer shots conditional on the historical IBM data; only AWS measurements can establish accuracy on the new devices.

**Selected collection · $195.6016 in QPU charges.** The Rigetti design samples GHZ, BV, QFT, PE, Grover and random volumetric circuits. It assigns 18 configurations to training and six to test, with two repetitions in each of two windows. A later window runs two repetitions on six validation configurations (two anchors, three new circuits and one larger random circuit). IonQ samples a matched subset once, with seven train, three test and four validation tasks. Its four validation circuits comprise an anchor, two unseen configurations and a larger random circuit. Rigetti costs $79.4016 and IonQ costs $116.20. The new schedule leaves $4.3984 of the $200 cap. Including the archived pilot at its recorded estimate leaves $3.6632. These small IonQ test and validation groups support per-circuit errors and intervals; they give little precision for estimating a device-specific model’s R². The larger-circuit label refers to the logical design. A native hardware gate-count extrapolation claim would require a verified final compilation.

Keep a configuration and all its repetitions on the same side of any development split, including across providers. Validation anchors deliberately repeat train configurations in a later window. Schedule stage two and validation on later dates and inspect provider snapshots for evidence of recalibration. A change in the calendar alone cannot establish a calibration change.

In [ ]:
# Review the per-task manifest before preparing anything.
planned = study.plan()
overview = pd.DataFrame(planned)
overview["QPU USD"] = [float(aws.task_price(study.DEVICES[r["vendor"]], r["shots"])) for r in planned]
print("Full campaign USD", study.estimate(planned))
print(overview.groupby(["vendor","stage","split"]).agg(tasks=("key","count"),
      configurations=("config","nunique"), shots_per_task=("shots","first"), usd=("QPU USD","sum")))
display(overview[["vendor", "stage", "split", "group", "algorithm", "config", "repetition", "shots"]])

## 3. Verify offline recovery on a real Rigetti result

This cell loads the archived circuit and the raw Braket response from the completed pilot. It rebuilds the histogram, computes the same four targets, and compares them to the saved analysis. You can perform the CSV export on another computer after pulling the commit containing the records.

In [ ]:
from qiskit import qpy
from experiments import aws_pilot_data as data

pilot = ROOT / "experiments/data/aws_pilot/bv3-alt-r1"
p, j = aws.load(pilot)
with (pilot / "logical.qpy").open("rb") as stream:
    circuit = qpy.load(stream)[0]
counts = data.decode_result(aws.read_json(pilot / "raw_result.json"), p)
rebuilt = data.evaluate(counts, p, circuit)
saved = aws.read_json(pilot / "result.json")
for target in ("success_rate", "dsr", "hellinger_fidelity", "tvd"):
    assert rebuilt["targets"]["fidelity"][target] == saved["targets"]["fidelity"][target], target
print("BV3-ALT:", counts["101"], "successes in", sum(counts.values()), "shots")

## 4. Prepare the selected schedule offline

Use the same collection branch and pull its latest commit before resuming. The manifest and frozen predictions live in `experiments/data/braket-repetitions-v1/`. Each task directory under `experiments/<algorithm>/data/qpu/aws/braket-repetitions-v1/` contains its request, circuits, journal and recovered records. The stage counts are 48 Rigetti and 10 IonQ tasks in development window 1, 48 Rigetti tasks in development window 2, and 12 Rigetti plus 4 IonQ tasks in prospective validation. Keep one submitting checkout for the campaign. Local file locks coordinate processes on one machine. Commit and push checkpoints before another person takes over submission.

Supply two existing Braket result buckets in the device regions, a named AWS profile if applicable, and the funds available for this campaign **at its start**. Deduct any earlier QPU spending that counts toward the $200 ceiling, including the archived $0.7352 pilot if applicable. Keep the same starting allowance on resume. The notebook compares the cost of the entire frozen schedule to this allowance. The professor’s role needs Braket device/task read, create and tag permissions, STS identity access, and access to the existing S3 result buckets. Keep credentials in the local AWS credential chain.

In [ ]:
RUN = study.CAMPAIGN_DIR
PREPARE = False
AVAILABLE_QPU_USD = None  # starting allowance, e.g. "199.2648" if the earlier pilot counts
BUCKETS = {"rigetti": "", "ionq": ""}  # us-west-1 and us-east-1 respectively
AWS_PROFILE = None
STAGE = "development-1"   # then development-2, then validation

def braket(*arguments):
    study.main(["--run-dir", str(RUN), *arguments])

def cloud_args():
    return ["--stage", STAGE] + (["--profile", AWS_PROFILE] if AWS_PROFILE else [])

if PREPARE:
    if AVAILABLE_QPU_USD is None or not all(BUCKETS.values()):
        raise ValueError("Enter the starting QPU allowance and both regional buckets")
    braket("prepare", "--available-usd", str(AVAILABLE_QPU_USD),
           "--rigetti-bucket", BUCKETS["rigetti"], "--ionq-bucket", BUCKETS["ionq"])

## 5. Preflight and submit one stage

Preflight checks identity, region, bucket, device status, gates and shot limits. Inspect the account, device names, and remaining allowance before enabling submission. To verify live decoding, submit the first scheduled task on each device, recover those two records, then set FIRST_PER_DEVICE to False and submit the rest of the stage. The two checks count toward the schedule. You may stop the kernel while tasks queue. Repeat the setup cells with the same directory, stage and credentials on restart. For prospective validation, complete section 7 before submitting.

In [ ]:
PREFLIGHT = False
if PREFLIGHT:
    braket("preflight", *cloud_args())

In [ ]:
# Paid operation. Type the stage name when ready to submit.
SUBMIT_STAGE = None
FIRST_PER_DEVICE = True
if SUBMIT_STAGE is not None:
    if SUBMIT_STAGE != STAGE or AVAILABLE_QPU_USD is None:
        raise ValueError("Check the stage and starting allowance")
    braket("submit", *cloud_args(), "--available-usd", str(AVAILABLE_QPU_USD),
           *(["--first-per-device"] if FIRST_PER_DEVICE else []))

## 6. Wait, recover and export

Poll while the kernel is open, or interrupt and rerun this cell later. Recovery reads task state and downloads completed results from S3. For a lost creation response, use the saved run ID tag to find the original task in the correct regional Braket console, then enter its ARN in `RECONCILED`. Keep failed and cancelled tasks for review. An interrupted submission can leave some prepared tasks; recover submitted ones and rerun the same submit stage for the remaining prepared requests.

In [ ]:
RECOVER = False
WAIT = False
RECONCILED = {}  # {task key: verified ARN} for an ambiguous submission
if RECOVER:
    tasks = [arg for key, arn in RECONCILED.items() for arg in ("--task", key, arn)]
    braket("recover", *cloud_args(), *(["--wait"] if WAIT else []), *tasks)

In [ ]:
EXPORT = False
if EXPORT:
    braket("export")  # also works after pulling a commit with recovered results

## 7. Freeze predictions before prospective submission

Commit and push the recovered development records. The analyst can pull that commit and run the export offline. Use the designated train rows for fitting and grouped tuning. Use the test rows for held-out evaluation; report test or validation errors by configuration so repetitions do not inflate the sample size. In a pooled analysis, group by configuration across devices. Use logical QWARD features (`a.*`) for common cross-provider predictors. Braket’s submitted features (`submitted.*`) describe a portable program; the provider compiles it again and its physical ISA and execution-time calibration are unknown. Existing IBM models that require native ISA or calibration features need a documented feature mapping or a retrained logical-feature model.

The following cell writes the exact keys required for prospective scoring. If the professor needs an end-to-end rehearsal or a simple prospective baseline, the cell can also compute a predeclared predictor: the mean observed target per algorithm and device across training configurations, weighting configurations equally. This baseline answers a different question from validation of your fitted QAQS models. Choose the model before viewing prospective results. Fill the template with predictions from the chosen frozen model for each target, and set its revision or artifact hash below. Commit and push the frozen predictions with the analysis revision, then pull that commit in the submitting checkout before returning to section 5 for the validation stage. This allows the professor to collect while the analyst owns modeling. The notebook checks that predictions cover every intended device/configuration and predate submission.

In [ ]:
WRITE_TEMPLATE = False
USE_TRAINING_BASELINE = False
FREEZE_MODEL_ID = None  # model revision for a filled validation-input.csv
if WRITE_TEMPLATE:
    braket("prediction-template")
if USE_TRAINING_BASELINE and FREEZE_MODEL_ID is not None:
    raise ValueError("Choose one prediction source")
if USE_TRAINING_BASELINE:
    braket("freeze-predictions", "--baseline")
elif FREEZE_MODEL_ID is not None:
    braket("freeze-predictions", "--model-id", FREEZE_MODEL_ID)

## 8. Commit the records and reproduce the CSV from another clone

Checkpoint after preparation, after submission has saved the task ARNs, and after recovery. Stop polling before staging the manifest, predictions and algorithm records from the repository root. After running the export, also stage the generated CSV. Review the staged paths and commit on the shared collection branch.

```sh
git add -- experiments/data/braket-repetitions-v1 ':(glob)experiments/*/data/qpu/aws/braket-repetitions-v1/**'
# After exporting:
git add -- experiments/dataset_out/braket-repetitions-v1/dataset.csv
git diff --cached --stat
git commit -m "data(braket): checkpoint recovered circuit measurements"
git push
```

In another clone, check out the collection branch, run `git pull --ff-only`, and run `python scripts/run.py braket export` with the project environment. This rebuilds `experiments/dataset_out/braket-repetitions-v1/dataset.csv` from the committed journals and raw measurements without AWS credentials. Each completed task has one row; the final schedule has 108 Rigetti and 14 IonQ tasks. Use `meta.split` to select training, test and validation rows. After prospective recovery, the score command compares measurements with the committed predictions for each device and configuration.

In [ ]:
SCORE = False
if SCORE:
    braket("score")